# Protein table — `protein.csv`

Verifies table 5. Proteomics values are already per-protein z-scored upstream (CCLE/Gygi), so `scale.scale_protein` is a documented no-op. The important check here isn't the values themselves but that `detected=False` (not-detected-by-mass-spec) and a `NaN` `zscore` always travel together — `docs/PROJECT_ARCHITECTURE.md` §3's abstain-never-impute rule depends on this.

In [ ]:
import pandas as pd

protein = pd.read_csv("../../data/processed/protein.csv")
protein.shape

In [ ]:
protein.head()

### `detected` vs. `zscore` — must never disagree
If a row is `detected=False` but has a real `zscore`, or `detected=True` with a missing `zscore`, that's a bug — the whole point of this flag is that `NaN` means "not detected", never "detected as zero".

In [ ]:
print(protein["detected"].value_counts())

mismatch = ((protein["detected"] == False) & protein["zscore"].notna()).sum() \
         + ((protein["detected"] == True) & protein["zscore"].isna()).sum()
print(f"\nMismatches between detected flag and zscore-null pattern: {mismatch}")

**Confirmed: 3,361,236 detected (77.0%), 1,007,889 not detected (23.0%), 0 mismatches** — every `detected=False` row has `zscore=NaN` and vice versa, exactly as designed. Absence of detection is kept distinct from a measured zero, never silently unified or imputed.

### Value range
A z-score should be roughly centered on 0 with unit-ish spread.

In [ ]:
protein.loc[protein["detected"], "zscore"].describe()

**Confirmed:** mean -0.029, std 0.95 — centered almost exactly on 0, as a z-score should be. The min of -23.1 is an extreme outlier but not implausible for a single protein/line pair in a panel-relative z-score (this is why proteomics is treated as an ordinal corroboration layer with its own floor/target, per `docs/PROJECT_ARCHITECTURE.md` §8, rather than assumed normally distributed).

### Verdict
The `detected`/`zscore` invariant holds with zero exceptions across 4.37M rows, and the value distribution looks like a genuine z-score. No concerns found.